Test integrating Pydantic with LlamaIndex. Use after inserting all the data with llamaindex_redis.ipynb

Load environment variables from .env file

In [2]:
import os

import nest_asyncio
from dotenv import load_dotenv

load_dotenv("../.env")
nest_asyncio.apply() # for async issues in Jupyter Notebook

Setup the embedding model

In [3]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [4]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: ]8;id=960679;https://logfire-us.pydantic.dev/iellis02/blue-horizon\https://logfire-us.pydantic.dev/iellis02/blue-horizon]8;;\

Connect to Redis Cloud

In [5]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.redis import RedisVectorStore
from redisvl.schema import IndexSchema

redis_conn_string = os.getenv("REDIS_URL")
schema = IndexSchema.from_dict(
    {
        "index": {"name": "blue_horizon", "prefix": "blue_horizon"},
        # customize fields that are indexed
        "fields": [
            # required fields for llamaindex
            {"type": "tag", "name": "id"},
            {"type": "tag", "name": "doc_id"},
            {"type": "text", "name": "text"},
            # custom vector field for bge-small-en-v1.5 embeddings
            {
                "type": "vector",
                "name": "vector",
                "attrs": {
                    "dims": 384,
                    "algorithm": "hnsw",
                    "distance_metric": "cosine",
                },
            },
        ],
    },
)
vector_store = RedisVectorStore(schema=schema, redis_url=redis_conn_string, overwrite=False)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

17:15:49 redisvl.index.index INFO   Index already exists, not overwriting.


In [6]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, storage_context=storage_context)
retriever = index.as_retriever(similarity_top_k=4)

In [7]:
retriever.retrieve("What swimming options are there?")

[NodeWithScore(node=TextNode(id_='FAQ000009', embedding=None, metadata={'category': 'amenities', 'subcategory': 'business', 'keywords': 'pool, swimming, recreation', 'last_updated': '2024-10-15'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nIs there a swimming pool?\n\nAnswer:\nYes, we have both indoor and outdoor pools open from 6:00 AM to 10:00 PM.', mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.6770186424260001),
 NodeWithScore(node=TextNode(id_='FAQ000010', embedding=None, metadata={'category': 'amenities', 'subcategory': 'recreation', 'keywords': 'spa, wellness, treatments', 'last_updated': '2024-01-07'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text='Question:\nDo you off

In [44]:
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings

model_settings = ModelSettings(temperature=0)

system_prompt = """You are an assistant who helps people find out information about a hotel.
Your sole job is to query the database for information about the hotel using the tool provided.
Provide the information returned from the tool that is relevant to the user's query
in a concise and well-formatted manner.

When you receieve the results from the tool, it may not return all the relevant results,
so mention that you are only listing some options.
You will not get different options by searching again, so do not offer to search for
more using the same query.

Assume that prices are in dollars.

Do not mention that you are searching a database, but you may mention that you are or
have perfomed a search.

Do not offer to do anything except search for information about the hotel, but not on
the same topic.

Do not provide any instructions to the user concerning the hotel that were not provided
to you.
"""

agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=system_prompt,
    model_settings=model_settings,
)

@agent.tool_plain
def query_hotel_info(query: str):
    """Provide information about the hotel in response to a passed-in query.
    The query should be concise and not ask for many details.
    """
    retrieved_nodes = retriever.retrieve(query)

    return [{"metadata": node.metadata, "text": node.text} for node in retrieved_nodes]

In [45]:
result = await agent.run("What dining options are there?")
print(result.output)

17:53:43.703 agent run
17:53:43.705   chat gpt-5-mini
17:53:46.675   running 1 tool
17:53:46.676     running tool: query_hotel_info
17:53:46.738   chat gpt-5-mini
Here are some dining options (listing some of the options returned by the search):

- Breakfast  
  - Included with most room rates. Please check your booking details.

- Room service  
  - Available 24/7. Full menu served during restaurant hours; limited menu overnight.

- Evening Dining  
  - Price: $65  
  - Duration/Service time: 45 minutes  
  - Availability: 8:00–20:00  
  - Location: Main Building  
  - Booking required: No  
  - Description: A 45-minute professionally presented dining experience to satisfy gourmet cravings.

- Dietary Specialist Menu  
  - Price: $75  
  - Duration/Service time: 45 minutes  
  - Availability: 6:00–22:00  
  - Location: Main Building  
  - Booking required: Yes (minimum notice: 0 hours)  
  - Description: A 45-minute specialized menu prepared by dietary experts for a memorable dining e

In [46]:
result = await agent.run("Where can I swim?")
print(result.output)

17:54:06.035 agent run
17:54:06.037   chat gpt-5-mini
17:54:09.806   running 1 tool
17:54:09.807     running tool: query_hotel_info
17:54:09.878   chat gpt-5-mini
I performed a search. Here are some options for swimming (listing some results):

- Pools
  - Indoor and outdoor pools
  - Hours: 6:00 AM – 10:00 PM

- Swim-related programs
  - Aqua Fitness Class
    - Price: $45
    - Duration: 45 minutes
    - Availability: 6:00 AM – 10:00 PM
    - Location: In-room
    - Booking required: No (min notice: 24 hours listed)
  - Personal Training Session (pool-based)
    - Price: $80
    - Duration: 45 minutes
    - Availability: By appointment only
    - Location: Pool Area
    - Booking required: No (min notice: 1 hour listed)

(Showing some of the available options returned by the search.)


In [47]:
result = await agent.run("I want to be pampered.")
print(result.output)

17:56:49.353 agent run
17:56:49.355   chat gpt-5-mini
17:56:53.547   running 1 tool
17:56:53.548     running tool: query_hotel_info
17:56:53.611   chat gpt-5-mini
I searched for spa and pampering options. Below are some options returned by the search (not exhaustive):

- Spa overview
  - Full-service spa offering massages, facials, and body treatments. Advance booking is recommended.

- Hotel amenities (includes)
  - Complimentary Wi‑Fi, fitness center, pool, spa, restaurant, 24‑hour room service.

- Couples Massage Experience
  - Price: $280
  - Duration: 90 minutes
  - Location: Spa & Wellness Center
  - Availability: By appointment only
  - Description: Signature couples treatment with therapeutic-grade materials and a customized approach to melt away stress and restore balance.

- Aromatherapy Journey
  - Price: $150
  - Duration: 75 minutes
  - Location: Pool Area
  - Availability: 06:00–22:00
  - Description: Aromatherapy-based relaxation using premium natural skincare products a

In [32]:
result = await agent.run("I want a massage.")
print(result.output)

17:43:42.844 agent run
17:43:42.846   chat gpt-5-mini
17:43:45.064   running 1 tool
17:43:45.065     running tool: query_hotel_info
17:43:45.132   chat gpt-5-mini
Here are the massage options available at the hotel's Spa & Wellness Center:

- Deep Tissue Massage
  - Price: $140
  - Duration: 60 minutes
  - Availability: 6:00–22:00
  - Booking required: No
  - Minimum notice: 4 hours

- Swedish Massage
  - Price: $120
  - Duration: 60 minutes
  - Availability: 24/7
  - Booking required: Yes
  - Minimum notice: 2 hours

- Couples Massage Experience
  - Price: $280
  - Duration: 90 minutes
  - Availability: By appointment only
  - Booking required: No
  - Minimum notice: 0 hours

General note: the hotel’s full-service spa offers massages, facials, and body treatments, and advance booking is recommended. These are the massage options listed in the hotel information.


# TODO: Fix this irrelevant info

In [49]:
result = await agent.run("Can I get some escargot?")
print(result.output)

17:58:46.902 agent run
17:58:46.904   chat gpt-5-mini
17:58:49.789   running 1 tool
17:58:49.790     running tool: query_hotel_info
17:58:49.860   chat gpt-5-mini
I searched for "escargot" — none of the items returned mention escargot specifically. Below are some related hotel info items from the search (only listing some options):

- Dining
  - Question: Is breakfast included?
  - Answer: Yes, breakfast is included with most room rates. Please check your booking details.

- Reservations
  - Question: How do I make a reservation?
  - Answer: You can make a reservation through our website, mobile app, or by calling our reservation desk. We accept all major credit cards.

- Payment policy
  - Question: What forms of payment do you accept?
  - Answer: We accept all major credit cards, debit cards, and cash with a security deposit.


Test GPT-5.1 (faster but doesn't follow instructions as well).

In [24]:
result = await agent.run("What dining options are there?")
print(result.output)

17:29:43.552 agent run
17:29:43.554   chat gpt-5.1
17:29:44.345   running 1 tool
17:29:44.345     running tool: query_hotel_info
17:29:44.407   chat gpt-5.1
Here are some of the dining options available at the hotel (this is only a selection of the options):

- **Hotel restaurants & breakfast**
  - Breakfast is **included with most room rates**. Please check your specific booking to confirm inclusion.

- **Room service**
  - **24/7 room service**  
    - Full menu available during restaurant hours  
    - Limited menu available overnight
  - **Evening Dining (Room Service Option)**  
    - Price: **$65**  
    - Approx. duration: **45 minutes** of service  
    - Availability: **08:00–20:00**  
    - Location: Main Building  
    - Booking required: **No**
  - **Dietary Specialist Menu (Room Service Option)**  
    - Price: **$75**  
    - Approx. duration: **45 minutes** of service  
    - Availability: **06:00–22:00**  
    - Location: Main Building  
    - Booking required: **Yes** 